# Canadian Grocery Inflation, 2015-2026

**Source:** Statistics Canada, Table 18-10-0004-01 — Consumer Price Index, monthly, not seasonally adjusted. Downloaded [here](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810000401).

This notebook explores how food prices have moved compared to overall inflation in Canada since January 2015. All series use the 2002=100 index base so they are directly comparable.

## Setup

In [ ]:
import pandas as pd
import numpy as np

from foodinflation.load import PRODUCTS
from foodinflation.analysis import ols, months_above

PROJECT_ROOT = pd.io.common.get_path(".")

df = pd.read_csv("data/processed/cpi_canada_food_2015_2026.csv", parse_dates=["date"])
wide = df.pivot(index="date", columns="product", values="value").sort_index()
print(f"{len(df)} rows, {df['date'].min():%Y-%m} to {df['date'].max():%Y-%m}")
df.head()

## Data quality check

Verify there are no missing values, that each product has a complete monthly series, and that index levels are positive.

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("\nMonths per product:")
print(df.groupby("product")["date"].agg(["count", "min", "max"]))
print("\nIndex levels all > 0:", (df["value"] > 0).all())
print("Min/Max value:", df["value"].min(), df["value"].max())

## Overall picture

How much has each category risen since January 2015 (index 2015-01 = 100)?

In [ ]:
base = wide.loc["2015-01-01"]
latest = wide.loc["2026-06-01"]
growth = (latest / base - 1) * 100
growth_sorted = growth.sort_values(ascending=False)
print("Cumulative index growth, Jan 2015 -> Jun 2026 (%):")
print(growth_sorted.round(1).to_string())

# Annualized (CAGR)
years = (wide.index[-1] - wide.index[0]).days / 365.25
cagr = ((latest / base) ** (1 / years) - 1) * 100
print("\nAnnualized (CAGR, %):")
print(cagr.sort_values(ascending=False).round(2).to_string())

## Year-over-year inflation by category

The year-over-year percent change is the standard way inflation is reported. Compare food purchased from stores against all-items.

In [ ]:
yoy = wide.pct_change(12) * 100
print("Latest YoY (%):")
print(yoy.tail(1).T.sort_values(by=yoy.index[-1], ascending=False).round(1).to_string())
print("\nPeak YoY by category (%):")
print(yoy.max().sort_values(ascending=False).round(1).to_string())

## Food vs overall inflation

How often has food-store inflation exceeded all-items inflation, and by how much?

In [ ]:
food = wide["Food purchased from stores"]
all_items = wide["All-items"]
food_yoy = yoy["Food purchased from stores"]
all_yoy = yoy["All-items"]

above = months_above(food_yoy, all_yoy)
total_months = food_yoy.notna().sum()
print(f"Months food-store inflation > all-items: {len(above)} / {total_months} ({(len(above)/total_months)*100:.0f}%)")
print(f"Average food-store inflation (all months): {food_yoy.mean():.2f}%")
print(f"Average all-items inflation: {all_yoy.mean():.2f}%")

## The 2021-2023 spike

Food inflation peaked in late 2022. How large was the gap, and where is it now?

In [ ]:
gap = food_yoy - all_yoy
print("Widest food vs all-items gap (percentage points):")
print(gap.idxmax().date(), round(gap.max(), 1))
print("\nLatest gap:", round(gap.iloc[-1], 1), "pp")
print("\nGap in 2022 (peak year):")
print(gap["2022"].describe().round(2).to_string())

## Regression: groceries vs all-items

A simple linear model of food-store inflation against all-items inflation. An r² near 1 means grocery prices move closely with the overall basket.

In [ ]:
mask = food_yoy.notna() & all_yoy.notna()
x = all_yoy[mask].values
y = food_yoy[mask].values
slope, intercept, r2 = ols(x, y)
print(f"food_yoy = {slope:.2f} * all_yoy + {intercept:.2f}")
print(f"r² = {r2:.3f}")
print(f"n = {len(x)}")
print("\nInterpretation: for every 1 percentage point the all-items CPI rises, grocery")
print("inflation rises by ~{:.2f} points on average.".format(slope))

## Takeaways

- Food purchased from stores has outrun overall inflation for most of the past decade.
- The gap widened sharply during the 2021-2023 inflation spike and has narrowed since.
- Fresh vegetables and fruit are the most volatile grocery categories.
- See the README and report for the full write-up.